# 06 · Asset Structure Principles and Content Aggregation

> Companion notebook to `docs/asset-structure/` from the Learn OpenUSD curriculum.

## Learning objectives
- Implement the four key principles of scalable asset structures: legibility, modularity, performance, navigability.
- Create asset entry points, encapsulate dependencies, and organize prim hierarchies with scopes.
- Model parallel user and computational workstreams using layer stacks and sublayers.
- Parameterize assets with primvars and variant sets, then loft those parameters above payloads.
- Apply the reference/payload pattern and build valid model hierarchies with `component`, `assembly`, and `group` kinds.

## How to use this notebook
This is a large module — use the Table of Contents in Jupyter (View → Table of Contents, or the sidebar) to navigate between sections `6.1` through `6.5`. Run cells top-to-bottom within a section. Generated files land in `./_assets/` next to this notebook. Safe to delete that folder any time.

In [1]:
import sys
print("Python:", sys.version.split()[0])
from pxr import Usd, Sdf, Gf, UsdGeom, UsdShade
print("pxr import OK.")
from pathlib import Path
NB_DIR = Path.cwd()
(NB_DIR / "_assets").mkdir(exist_ok=True)
print("_assets/ ready at:", NB_DIR / "_assets")

Python: 3.12.9
pxr import OK.
_assets/ ready at: /Users/stranzersweb/Projects/LearnOpenUSD/notebooks/.exec/06_asset_structure/_assets


In [2]:
try:
    from lousd.utils.helperfunctions import create_new_stage
    from lousd.utils.visualization import DisplayUSD, DisplayCode
    print("lousd helpers loaded.")
    _HAVE_LOUSD = True
except ImportError:
    import os
    from pathlib import Path as _P
    def create_new_stage(relative_file_path: str):
        from pxr import Usd, Sdf
        ident = os.path.join(os.getcwd(), relative_file_path)
        found = Sdf.Layer.Find(identifier=ident)
        if found:
            return Usd.Stage.Open(found.identifier)
        return Usd.Stage.CreateNew(relative_file_path)
    def DisplayUSD(path, show_usd_code=True):
        from pxr import Usd
        print(f"--- {path} ---")
        print(Usd.Stage.Open(str(path)).ExportToString(addSourceFileComment=False))
    def DisplayCode(path):
        print(f"--- {path} ---")
        print(_P(path).read_text())
    print("Using fallback helpers.")
    _HAVE_LOUSD = False

lousd helpers loaded.


In [3]:
# Safely stage the module's exercise content under _assets/ so the rest of the
# notebook can run end-to-end. If the source folder is missing we continue
# with best-effort fallbacks (individual cells check for specific files).
import shutil
from pathlib import Path

SRC_EX = Path("../docs/exercise_content/asset_structure").resolve()
DST_EX = (Path.cwd() / "_assets" / "asset_structure").resolve()

try:
    if SRC_EX.exists():
        if DST_EX.exists():
            shutil.rmtree(DST_EX)
        shutil.copytree(SRC_EX, DST_EX)
        print(f"Copied exercise content to {DST_EX}")
    else:
        print(f"WARNING: exercise source not found at {SRC_EX}. Cells that depend on it will be skipped.")
except Exception as e:
    print(f"WARNING: could not copy exercise content: {e}")

HAVE_EX = DST_EX.exists()
print("HAVE_EX =", HAVE_EX)

HAVE_EX = False


## 6.1 Principles of Asset Structure

An **asset** is a named, versioned, structured container of resources — composable USD layers, textures, volumetric data, and more. Asset structure lets those resources be reused, collaborated on, and streamed at scale. There is no one-size-fits-all layout; structures have to be tailored to the domain. But scalable asset structures share four principles:

- **Legibility** — names and conventions clearly convey intent and type.
- **Modularity** — clear entry points and stable interfaces enable iterative reuse.
- **Performance** — read/write costs are predictable; reference/payload pairs draw boundaries around expensive data.
- **Navigability** — prim hierarchies, paths, and relationships make discovery easy.

The rest of this section turns these principles into concrete code against a running example: a building asset named `lrg_bldgF`.

### 6.1.1 Your first asset — creating an entry point

A raw export often has sibling root prims (meshes, materials, …) with no common ancestor. Without a single entry point, a downstream user has no single path to reference and bring the whole thing along. The fix is to define an `Xform` (often called `/World`) and reparent every existing root prim underneath it — `Usd.NamespaceEditor` takes care of updating relationship targets like material bindings.

In [4]:
# Recreates docs/asset-structure/asset-structure-principles/your-first-asset.md
from pathlib import Path
import shutil
from pxr import Usd, UsdGeom, Sdf

ex1 = Path("_assets/asset_structure/exercise_01")
if not ex1.exists():
    print("Skipping 6.1.1 — exercise_01 not available.")
else:
    src = ex1 / "export.usd"
    working = Path("_assets/06_01_entry_point.usd")
    shutil.copy(src, working)

    stage = Usd.Stage.Open(str(working))
    root_prims = list(stage.GetPseudoRoot().GetChildren())
    print("Original root prims:", [p.GetPath() for p in root_prims])

    # Define a new World entry point and make it the default prim.
    world_prim = UsdGeom.Xform.Define(stage, "/World").GetPrim()
    stage.SetDefaultPrim(world_prim)

    # Move every old root prim under /World. NamespaceEditor fixes up
    # relationship targets (e.g., material:binding) as it goes.
    editor = Usd.NamespaceEditor(stage)
    for prim in root_prims:
        editor.ReparentPrim(prim, world_prim)
        editor.ApplyEdits()

    stage.GetRootLayer().Save()
    print("\nAfter reparenting:")
    for p in stage.Traverse():
        print(" ", p.GetPath(), "(", p.GetTypeName() or "<untyped>", ")")

Skipping 6.1.1 — exercise_01 not available.


In [5]:
# Preview: everything now hangs off a single referenceable /World entry point.
from pathlib import Path
p = Path("_assets/06_01_entry_point.usd")
if p.exists():
    DisplayUSD(p)
else:
    print("Preview skipped (file missing).")

Preview skipped (file missing).


**What just happened:** with a single entry point `/World` set as `defaultPrim`, the asset can now be brought into any scene with a single `AddReference` call and every mesh + material comes along.

### 6.1.2 Encapsulation — every dependency under the entry point

Encapsulation means *everything* the asset needs — including relationship targets like material bindings — must be a descendant of the entry point. In the original export, the `roof.material` lived outside `/World`; when the asset was referenced three times into a scene, the binding targets pointed to paths outside the reference scope and OpenUSD silently dropped them, leaving white roofs. The fix is the same pattern: reparent the stray prims beneath `/World`.

In [6]:
# Build a tiny stage that mimics the encapsulation bug and then fixes it.
from pathlib import Path
from pxr import Usd, UsdGeom, UsdShade, Sdf

bad_path = Path("_assets/06_02_unencapsulated.usda")
stage = create_new_stage(str(bad_path))

world = UsdGeom.Xform.Define(stage, "/World")
stage.SetDefaultPrim(world.GetPrim())
mesh = UsdGeom.Mesh.Define(stage, "/World/roof")
# Material lives OUTSIDE of /World - the bug.
material = UsdShade.Material.Define(stage, "/roof_material")
UsdShade.MaterialBindingAPI.Apply(mesh.GetPrim()).Bind(material)
stage.GetRootLayer().Save()

# Now fix by reparenting the material under /World.
editor = Usd.NamespaceEditor(stage)
editor.ReparentPrim(stage.GetPrimAtPath("/roof_material"), world.GetPrim())
editor.ApplyEdits()
stage.GetRootLayer().Save()

print("After encapsulation fix:")
for p in stage.Traverse():
    print(" ", p.GetPath())

After encapsulation fix:
  /World
  /World/roof
  /World/roof_material


In [7]:
DisplayUSD("_assets/06_02_unencapsulated.usda")

**What just happened:** every prim the asset depends on now lives under `/World`. When the asset is referenced elsewhere, OpenUSD no longer emits "target outside the scope of the reference" warnings and the material binding survives.

### 6.1.3 Organizing the prim hierarchy with Scopes

`Scope` prims are pure organizational containers — unlike `Xform` they don't carry transforms, so they don't muddle semantics. A common partition separates `Geometry` (modelers) from `Looks` (surfacers), preventing namespace collisions and making deactivation of an entire subtree a one-liner.

In [8]:
# Adapts docs/asset-structure/asset-structure-principles/organizing-prim-hierarchy.md
from pathlib import Path
import shutil
from pxr import Usd, UsdGeom, UsdShade, Sdf

src_ex = Path("_assets/asset_structure/exercise_03")
target = Path("_assets/06_03_organized.usd")
if src_ex.exists() and (src_ex / "lrg_bldgF.usd").exists():
    shutil.copy(src_ex / "lrg_bldgF.usd", target)
    # Bring over the contents folder too (used as sublayer).
    if (src_ex / "contents").exists():
        contents_dst = Path("_assets/contents_06_03")
        if contents_dst.exists():
            shutil.rmtree(contents_dst)
        shutil.copytree(src_ex / "contents", contents_dst)
    stage = Usd.Stage.Open(str(target))
else:
    # Fallback: build a minimal stage with mixed mesh/material children.
    stage = create_new_stage(str(target))
    world = UsdGeom.Xform.Define(stage, "/World")
    stage.SetDefaultPrim(world.GetPrim())
    UsdGeom.Mesh.Define(stage, "/World/door")
    UsdGeom.Mesh.Define(stage, "/World/roof")
    UsdShade.Material.Define(stage, "/World/door_mat")
    UsdShade.Material.Define(stage, "/World/roof_mat")

default_prim = stage.GetDefaultPrim()
geom_scope = UsdGeom.Scope.Define(stage, default_prim.GetPath().AppendChild("Geometry")).GetPrim()
looks_scope = UsdGeom.Scope.Define(stage, default_prim.GetPath().AppendChild("Looks")).GetPrim()

editor = Usd.NamespaceEditor(stage)
for prim in list(default_prim.GetChildren()):
    if prim in (geom_scope, looks_scope):
        continue
    if prim.IsA(UsdGeom.Mesh):
        editor.ReparentPrim(prim, geom_scope)
        editor.ApplyEdits()
    elif prim.IsA(UsdShade.Material):
        editor.ReparentPrim(prim, looks_scope)
        editor.ApplyEdits()

stage.GetRootLayer().Save()
print("After Scope partitioning:")
for p in stage.Traverse():
    print(" ", p.GetPath(), "(", p.GetTypeName() or "<untyped>", ")")

After Scope partitioning:
  /World ( Xform )
  /World/Geometry ( Scope )
  /World/Geometry/door ( Mesh )
  /World/Geometry/roof ( Mesh )
  /World/Looks ( Scope )
  /World/Looks/door_mat ( Material )
  /World/Looks/roof_mat ( Material )


In [9]:
DisplayUSD("_assets/06_03_organized.usd")

**What just happened:** meshes and materials are now grouped under `Geometry` and `Looks` scopes. Downstream consumers can deactivate one or the other cleanly, and jurisdiction between modelers and surfacers is obvious at a glance.

## 6.2 Workstreams with Layer Stacks

Real assets are rarely a single file. Teams need to iterate in parallel: a modeler on geometry, a surfacer on materials, a rigger on controls. OpenUSD models this with **layer stacks** — the root layer plus its sublayers, composed in order. Each workstream authors sparsely into its own layer.

Workstreams can also be **computational** (a simulation partitioned across processes) or **hybrid** (synthetic motion on top of hand-authored keys). The important rule: layer stacks are not a versioning system. Keep them small and bounded, not procedurally growing.

### 6.2.1 Sparse sublayer for a shading workstream

Below we simulate a surfacing artist exporting only material + binding data into `shading.usd`, then adding it as a sublayer to the asset root. A second run with a different `diffuseColor` shows how two artists can iterate without clobbering each other.

In [10]:
# Adapts docs/asset-structure/workstreams/adding-user-workstreams.md
from pathlib import Path
from pxr import Usd, UsdGeom, UsdShade, Sdf

asset_path = Path("_assets/06_04_lrg_bldgF.usda")
shading_path = Path("_assets/06_04_shading.usda")

# 1. Create the asset root with a minimal geometry workstream baked in.
stage = create_new_stage(str(asset_path))
world = UsdGeom.Xform.Define(stage, "/World").GetPrim()
stage.SetDefaultPrim(world)
geom = UsdGeom.Scope.Define(stage, "/World/Geometry")
UsdGeom.Mesh.Define(stage, "/World/Geometry/roof")
UsdGeom.Scope.Define(stage, "/World/Looks")
stage.GetRootLayer().Save()

def export_sparse_materials(diffuse_color):
    """Simulate a surfacing DCC sparse export into shading.usda."""
    shading = Usd.Stage.CreateNew(str(shading_path)) if not shading_path.exists() else Usd.Stage.Open(str(shading_path))
    mat_path = "/World/Looks/roof"
    mat = UsdShade.Material.Define(shading, mat_path)
    shader = UsdShade.Shader.Define(shading, mat_path + "/PBR")
    shader.CreateIdAttr("UsdPreviewSurface")
    shader.CreateInput("diffuseColor", Sdf.ValueTypeNames.Color3f).Set(diffuse_color)
    mat.CreateSurfaceOutput().ConnectToSource(shader.ConnectableAPI(), "surface")
    # Bind to the roof mesh.
    roof = shading.OverridePrim("/World/Geometry/roof")
    UsdShade.MaterialBindingAPI.Apply(roof).Bind(mat)
    shading.GetRootLayer().Save()

    # Ensure shading.usda is a sublayer of the asset root.
    root_layer = Sdf.Layer.FindOrOpen(str(asset_path))
    rel = "./" + shading_path.name
    if rel not in root_layer.subLayerPaths:
        root_layer.subLayerPaths.append(rel)
        root_layer.Save()

# First surfacing pass - green roof.
export_sparse_materials((0.337, 0.737, 0.6))
# Second pass from the same artist - blue roof. Sparse override, no rework needed.
export_sparse_materials((0.0, 0.0, 1.0))

print("Sublayers on asset root:", Sdf.Layer.FindOrOpen(str(asset_path)).subLayerPaths)

Sublayers on asset root: ['./06_04_shading.usda']


In [11]:
DisplayCode("_assets/06_04_lrg_bldgF.usda")
DisplayCode("_assets/06_04_shading.usda")

**What just happened:** the asset root stays untouched while the surfacing layer evolves independently. Two artists could now iterate at the same time without conflicts — one owns `lrg_bldgF.usda`, the other owns `shading.usda`.

## 6.3 Asset Parameterization

Parameterization lets a single asset vary downstream without duplicating files. OpenUSD offers two primary mechanisms:

- **Variant sets** — discrete, named alternatives (`red` / `blue` / `green`). Ideal when the variations are fundamentally different topology, materials, or LODs. Each selection creates a new instancing prototype.
- **Primvars** — continuous parameters inherited down the prim hierarchy. Ideal for single scalar/color values read by shaders (think `asset_base_color`). No new prototypes.

Primvars are lighter — memory savings upfront at the cost of an extra material lookup. Authored at the entry point, they become public, editable controls.

### 6.3.1 Authoring a primvar as an asset parameter

In [12]:
# Adapts docs/asset-structure/asset-parameterization/exercise-asset-parameterization.md
from pathlib import Path
from pxr import Usd, UsdGeom, Sdf

asset = Path("_assets/06_05_param_asset.usda")
stage = create_new_stage(str(asset))
world = UsdGeom.Xform.Define(stage, "/World").GetPrim()
stage.SetDefaultPrim(world)

# Create a primvar on the entry point. A downstream shader graph (a
# UsdPrimvarReader_float3 with the name 'accentColor') would read this
# and fall back to a default if nothing is authored.
primvars_api = UsdGeom.PrimvarsAPI(world)
accent_color = primvars_api.CreatePrimvar(
    "accentColor",
    Sdf.ValueTypeNames.Float3,
    UsdGeom.Tokens.constant,
)
accent_color.Set((1.0, 0.0, 0.0))  # public, downstream-editable
stage.GetRootLayer().Save()

print("accentColor =", accent_color.Get())
print("interpolation =", accent_color.GetInterpolation())

accentColor = (1, 0, 0)
interpolation = constant


In [13]:
DisplayUSD("_assets/06_05_param_asset.usda")

**What just happened:** `primvars:accentColor` authored on `/World` is inherited down to every descendant mesh. Any shader wired to a `UsdPrimvarReader_float3` reading `accentColor` will pick it up, and a downstream scene can override the value without touching the asset's internals.

## 6.4 The Reference / Payload Pattern

Large assets shouldn't force every downstream user to know whether they need a payload. The **reference/payload pattern** solves this: the asset's interface file is designed to be *referenced*, and the payload is an internal implementation detail hidden behind the interface. Downstream users always `AddReference` — they never think about payloads.

The interface layer is cheap: it holds metadata, a default prim, and a payload arc pointing at `contents.usd`. When the payload is unloaded, the interface remains.

### 6.4.1 Splitting an asset into interface + contents

In [14]:
# Adapts docs/asset-structure/reference-payload-pattern/exercise-ref-payload-pattern.md
from pathlib import Path
from pxr import Usd, UsdGeom, Sdf

working = Path("_assets").resolve()
asset_name = "06_06_lrg_bldgF.usda"
contents_name = "06_06_contents.usda"

# Start from the parameterized asset above as the 'pre-refactor' state.
src = working / "06_05_param_asset.usda"
pre_refactor = working / asset_name
import shutil as _sh
if src.exists():
    _sh.copy(src, pre_refactor)
else:
    Usd.Stage.CreateNew(str(pre_refactor)).GetRootLayer().Save()

# PART 1: "Save As" the current asset root as contents.usda.
stage = Usd.Stage.Open(str(pre_refactor))
asset_layer = stage.GetRootLayer()
asset_layer.Export(str(working / contents_name), args={"format": "usda"})

# PART 2: Rebuild a thin asset root whose only job is to payload contents.
iface = Usd.Stage.CreateInMemory()
world_prim = UsdGeom.Xform.Define(iface, "/World").GetPrim()
iface.SetDefaultPrim(world_prim)
UsdGeom.SetStageMetersPerUnit(iface, UsdGeom.LinearUnits.centimeters)
UsdGeom.SetStageUpAxis(iface, UsdGeom.Tokens.y)
world_prim.GetPayloads().AddPayload(f"./{contents_name}")
iface.GetRootLayer().Export(str(pre_refactor), args={"format": "usda"})

print("Interface layer (thin) is at:", pre_refactor)
print("Contents (heavy) is at:", working / contents_name)

Interface layer (thin) is at: /Users/stranzersweb/Projects/LearnOpenUSD/notebooks/.exec/06_asset_structure/_assets/06_06_lrg_bldgF.usda
Contents (heavy) is at: /Users/stranzersweb/Projects/LearnOpenUSD/notebooks/.exec/06_asset_structure/_assets/06_06_contents.usda


In [15]:
DisplayCode("_assets/06_06_lrg_bldgF.usda")

**What just happened:** the root layer is now a tiny stub that payloads `contents.usda`. Downstream scenes still reference `lrg_bldgF.usda` as before, but can now *unload* the payload to explore the scenegraph cheaply.

### 6.4.2 Lofting primvars above the payload

There is a catch: the moment we moved everything behind a payload, the `primvars:accentColor` we carefully advertised in 6.3 is only visible when the payload is *loaded*. If a downstream consumer unloads the payload to save memory, they lose discoverability of the parameter.

The fix is **lofting**: copy selected fields from the contents layer up into the interface layer so they remain visible even with the payload unloaded. We use the lower-level `Sdf` API because we want to move specs around on a per-layer basis, bypassing composition.

In [16]:
# Adapts docs/asset-structure/reference-payload-pattern/lofting-primvars.md
from pathlib import Path
from pxr import Sdf

asset_layer = Sdf.Layer.FindOrOpen(str(Path("_assets/06_06_lrg_bldgF.usda").resolve()))
contents_layer = Sdf.Layer.FindOrOpen(str(Path("_assets/06_06_contents.usda").resolve()))

if asset_layer and contents_layer:
    prim_spec = contents_layer.GetPrimAtPath("/World")
    if prim_spec is not None:
        # Copy every property spec from contents /World up to the interface.
        for prop_spec in list(prim_spec.properties):
            Sdf.CopySpec(contents_layer, prop_spec.path, asset_layer, prop_spec.path)
            prim_spec.RemoveProperty(prop_spec)
        asset_layer.Save()
        contents_layer.Save()
    print("Lofted properties into asset layer.")
else:
    print("Skipping lofting — layers not available.")

Lofted properties into asset layer.


In [17]:
DisplayCode("_assets/06_06_lrg_bldgF.usda")

**What just happened:** `primvars:accentColor` now lives in the interface layer. With the payload unloaded, downstream code can still enumerate the property and set a value — the parameter is discoverable without ever opening the heavy contents.

### 6.4.3 Lofting variant sets

The same idea applies to variant sets. If an `exteriorType` variant set lives deep inside `contents.usd` on the `Looks` scope, downstream users can't see it when the payload is unloaded. Lofting creates a matching variant set on the asset entry point whose only job is to *select* the matching variant in the inner set — a tiny forwarder.

Conceptually:
```python
lofted_vset = default_prim.GetVariantSets().AddVariantSet("exteriorType")
for variant in variants:
    lofted_vset.AddVariant(variant)
    lofted_vset.SetVariantSelection(variant)
    with lofted_vset.GetVariantEditContext():
        inner_vset.SetVariantSelection(variant)
lofted_vset.SetVariantSelection(default)
```
Each variant on the lofted set authors a single opinion — "set the inner `exteriorType` variant to the matching name" — inside its own edit context. This section is summarized rather than executed because it depends on variant sets authored elsewhere in the exercise pack; the code pattern above is the load-bearing part.

## 6.5 Model Hierarchy

A full scenegraph can contain millions of prims. **Model hierarchy** is a shallower, curated view of it built from `kind` metadata. It lets algorithms and users prune traversal at well-defined boundaries:

- **`component`** — the "leaf" of the model hierarchy. Think consumer-facing product: a pen, a house, a building. Cannot contain other components.
- **`assembly`** — an important group, usually an aggregate asset (a neighborhood of houses, a city block).
- **`group`** — an intermediate organizational prim between assemblies and components.
- **`subcomponent`** — annotation for important prims *below* a component, outside the model hierarchy proper.

Model hierarchies should be **operational** (self-contained and renderable), **shallow**, **consistent**, and **extensible** through naming conventions (e.g., `c_` / `a_` prefixes) rather than custom kinds whenever possible.

### 6.5.1 Marking an asset as a Component

In [18]:
# Adapts docs/asset-structure/model-hierarchy/exercise-components.md
from pathlib import Path
from pxr import Usd, UsdGeom, Kind

component_path = Path("_assets/06_09_component.usda")
stage = create_new_stage(str(component_path))
world = UsdGeom.Xform.Define(stage, "/World").GetPrim()
stage.SetDefaultPrim(world)
UsdGeom.Scope.Define(stage, "/World/Geometry")
UsdGeom.Scope.Define(stage, "/World/Looks")

# Tag the asset entry point as a component model.
Usd.ModelAPI(world).SetKind(Kind.Tokens.component)
stage.GetRootLayer().Save()

print("/World kind =", Usd.ModelAPI(world).GetKind())
print("IsComponent:", world.IsComponent())
print("IsModel:", world.IsModel())

/World kind = component
IsComponent: True
IsModel: True


In [19]:
DisplayUSD("_assets/06_09_component.usda")

**What just happened:** `/World` is now a valid `component` model. Traversal utilities that use `Usd.PrimRange` with model-hierarchy predicates will treat it as a pruning point.

### 6.5.2 Building an Assembly that references Components

In [20]:
# Adapts docs/asset-structure/model-hierarchy/exercise-assemblies.md
from pathlib import Path
from pxr import Usd, UsdGeom, Gf, Sdf, Kind

assembly_path = Path("_assets/06_10_city_blockA.usda")
stage = create_new_stage(str(assembly_path))
world = UsdGeom.Xform.Define(stage, "/World").GetPrim()
stage.SetDefaultPrim(world)

# PART 1: Mark the entry point as an assembly.
Usd.ModelAPI(world).SetKind(Kind.Tokens.assembly)

def position_bldg(prim, x):
    xf = UsdGeom.Xformable(prim)
    xf.ClearXformOpOrder()
    xf.AddTranslateOp().Set(Gf.Vec3d(x * 500.0, 0.0, 0.0))

# PART 2: Reference the component six times into the assembly.
for x in range(1, 7):
    ref_path = world.GetPath().AppendChild(f"lrg_bldgF_{x:02}")
    ref_prim = UsdGeom.Xform.Define(stage, ref_path).GetPrim()
    ref_prim.GetReferences().AddReference("./06_09_component.usda")
    position_bldg(ref_prim, x)

stage.GetRootLayer().Save()

print("/World kind =", Usd.ModelAPI(world).GetKind())
for child in world.GetChildren():
    print(" ", child.GetPath(), "kind =", Usd.ModelAPI(child).GetKind() or "<inherited>")

/World kind = assembly
  /World/lrg_bldgF_01 kind = component
  /World/lrg_bldgF_02 kind = component
  /World/lrg_bldgF_03 kind = component
  /World/lrg_bldgF_04 kind = component
  /World/lrg_bldgF_05 kind = component
  /World/lrg_bldgF_06 kind = component


In [21]:
DisplayUSD("_assets/06_10_city_blockA.usda")

**What just happened:** a valid two-level model hierarchy — `World (assembly)` containing six `lrg_bldgF_NN (component)` references. Because the kinds are set correctly, model-hierarchy-aware code can prune traversal at each building.

### 6.5.3 Adding intermediate Groups

In [22]:
# Adapts docs/asset-structure/model-hierarchy/exercise-groups.md
from pathlib import Path
from enum import Enum
from pxr import Usd, UsdGeom, Gf, Kind

class Side(Enum):
    North = 1
    South = -1

grouped = Path("_assets/06_11_city_blockA_groups.usda")
stage = create_new_stage(str(grouped))
world = UsdGeom.Xform.Define(stage, "/World").GetPrim()
stage.SetDefaultPrim(world)
Usd.ModelAPI(world).SetKind(Kind.Tokens.assembly)

def position_bldg(prim, x, side):
    xf = UsdGeom.Xformable(prim)
    xf.ClearXformOpOrder()
    xf.AddTranslateOp().Set(Gf.Vec3d(x * 500.0, 0.0, side.value * 400.0))

for side in Side:
    # Intermediate group prim between /World and the component references.
    side_xform = UsdGeom.Xform.Define(stage, world.GetPath().AppendChild(side.name))
    Usd.ModelAPI(side_xform).SetKind(Kind.Tokens.group)
    for x in range(1, 4):
        ref_path = side_xform.GetPath().AppendChild(f"lrg_bldgF_{x:02}")
        ref_prim = UsdGeom.Xform.Define(stage, ref_path).GetPrim()
        ref_prim.GetReferences().AddReference("./06_09_component.usda")
        position_bldg(ref_prim, x, side)

stage.GetRootLayer().Save()

for prim in stage.Traverse():
    kind = Usd.ModelAPI(prim).GetKind()
    if kind:
        print(" ", prim.GetPath(), "kind =", kind)

  /World kind = assembly
  /World/North kind = group
  /World/North/lrg_bldgF_01 kind = component
  /World/North/lrg_bldgF_02 kind = component
  /World/North/lrg_bldgF_03 kind = component
  /World/South kind = group
  /World/South/lrg_bldgF_01 kind = component
  /World/South/lrg_bldgF_02 kind = component
  /World/South/lrg_bldgF_03 kind = component


In [23]:
DisplayUSD("_assets/06_11_city_blockA_groups.usda")

**What just happened:** we now have `World (assembly) → North/South (group) → lrg_bldgF_NN (component)`. The scenegraph changed but the visuals didn't — groups are organizational, not transformative by semantics.

### 6.5.4 Variation workstream at the assembly level

The final exercise simulates a surfacing artist working *at the assembly context*, traversing component children, and authoring sparse variant selections + primvar overrides into a shading sublayer of `city_blockA`. The load-bearing code pattern:

```python
for prim in stage.Traverse():
    if prim.IsComponent() and prim.GetVariantSets().HasVariantSet("exteriorType"):
        UsdGeom.PrimvarsAPI(prim).GetPrimvar("accentColor").Set(random.choice(accents))
        prim.GetVariantSet("exteriorType").SetVariantSelection(random.choice(variants))
```

The traversal uses `IsComponent()` as its pruning test — exactly the kind of model-hierarchy shortcut we set up in 6.5.1–6.5.3. Because edits are made inside an edit target on the assembly's `contents/shading.usd` sublayer, the per-building components are never modified. This section is **summarized only** (not executed) because it depends on the full exterior variant set authored in the upstream exercise pack.

## Key Takeaways

- **An asset needs a single entry point.** A `defaultPrim` (commonly `/World`) gives downstream users one path to reference and keeps the asset self-contained.
- **Encapsulation is non-negotiable.** Everything the asset depends on — including relationship targets like material bindings — must live under the entry point.
- **Use `Scope` prims for organization**, not `Xform`. They carry no transform semantics, so partitions like `Geometry` / `Looks` stay clean.
- **Model workstreams as layer stacks.** Geometry and shading layers let artists iterate in parallel without clobbering each other, and the root layer stays sparse.
- **Parameterize with primvars for continuous values, variant sets for discrete alternatives.** Advertise public parameters on the entry point.
- **Hide complexity behind the reference/payload pattern.** Downstream scenes reference a thin interface; payloads load on demand. Loft important fields (primvars, variant sets, `extentsHint`) above the payload so they remain discoverable when the payload is unloaded.
- **Model hierarchy = the shallow, kind-tagged view of your scene.** `component` marks pruning leaves, `assembly` aggregates them, `group` provides intermediate organization. Keep hierarchies shallow, consistent, and operational.

## Next up

Continue with **`07_asset_modularity_instancing.ipynb`**, which builds on these structural principles to make thousands of asset references cheap through scene graph instancing and prototypes.